# Garbage Image Classification 

## Project overview

Proper waste segregation plays an important role in improving recycling efficiency and reducing environmental impact. However, identifying the correct category for different types of garbage can sometimes be difficult, especially for objects with similar appearance or mixed materials. The goal of this project is to develop an image classification system capable of automatically recognizing different categories of waste using deep learning techniques.

This project applies transfer learning with an EfficientNet-based image classification model to classify garbage images into **11 categories**:

- Aluminium  
- Batteries  
- Cardboard  
- Disposable Plates  
- Glass  
- Hard Plastic  
- Paper  
- Paper Towel  
- Polystyrene  
- Soft Plastics  
- Takeaway Cups  

The project includes data preprocessing, stratified train–test splitting, 5-fold cross-validation, model comparison across image resolutions (224×224 and 448×448), and evaluation on both real-world images and unseen test data.

## Busines context

Waste management and recycling systems rely heavily on accurate waste segregation. Incorrect disposal of recyclable materials can increase processing costs, reduce recycling efficiency, and lead to contamination of entire waste streams. Manual classification is often inefficient and subject to human error, particularly when dealing with large volumes of waste.

An automated image classification system can support smarter waste management by assisting users in identifying the correct disposal category. Such a solution could be integrated into mobile applications, smart bins, or automated sorting systems to provide real-time recommendations. From a business perspective, improving classification accuracy may reduce operational costs, increase recycling effectiveness, and contribute to environmental sustainability initiatives.

## Objectives

The main objectives of this project are:

- Investigate the impact of image resolution on model performance
- Evaluate model generalization using unseen data
- Compare performance across different testing scenarios
- Analyze strengths and weaknesses of the trained classifier

## Workflow

1. Load and preprocess image dataset  
2. Create train and hold-out test split  
3. Perform 5-fold cross-validation  
4. Train EfficientNet-based classifiers  
5. Compare 224×224 and 448×448 models  
6. Test on real-world images  
7. Evaluate on unseen hold-out dataset  
8. Analyze results and model behavior

## Expected outcome

The expected outcome is a model capable of identifying garbage categories with reasonable accuracy while maintaining good generalization performance on previously unseen examples.

## Authors
Yuliya Martyniuk 474075
Weronika Mądro 473193

# Libaries upload

The first stage of the project focused on importing the required libraries and preparing the dataset for further processing.  Next, the image dataset was loaded from the local trash_images directory using the imagefolder format. This approach automatically reads folder names as class labels and creates a structured dataset suitable for image classification tasks.

To ensure a reliable evaluation process, the dataset was split using stratified sampling. A 15% hold-out subset was separated and reserved as an unseen test set, while the remaining 85% of the images were assigned for model development and cross-validation. Stratification ensured that the class distribution remained consistent across subsets, reducing the risk of introducing bias caused by imbalanced category proportions. A fixed random seed was used to guarantee reproducibility, allowing the exact same train-test split to be recreated in future experiments.

Finally, label mappings were generated. Category names were converted into numerical identifiers and corresponding reverse mappings (id2label) were created. This step allows the model to work with numerical outputs internally while still making predictions interpretable by converting predicted indices back into human-readable class names.

In [ ]:
# !pip install datasets
# !pip install imagehash
# !pip install torchvision
# !pip install transformers
# !pip install evaluate
# !pip install transformers[torch]
#!pip install "accelerate>=1.1.0" 

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from datasets import load_dataset
from huggingface_hub import login
from collections import Counter
from PIL import Image
import imagehash
import numpy as np
from transformers import AutoImageProcessor, AutoModelForImageClassification, TrainingArguments, Trainer
from torchvision.transforms import Compose, RandomHorizontalFlip, RandomRotation, ColorJitter
import evaluate
import torch
from sklearn.model_selection import StratifiedKFold
import gc
import os
import json


In [ ]:
# login(token="") 
# dataset = load_dataset("viola77data/recycling-dataset",num_proc=8)
# data = dataset["train"] # the author named the whole dataset as train, the 'train' split contains all the data in this case

# total_images = len(data)
# print(f"Loaded {total_images:,} images locally.\n")

In [ ]:
# data_dir = r"C:\Users\ydmar\.cache\huggingface\hub\datasets--viola77data--recycling-dataset\snapshots\e2e03c91c385e8d1a758389cdb20cf9c024f6cbf"
data_dir = r"./trash_images" 
dataset = load_dataset("imagefolder", data_dir=data_dir)
data = dataset["train"]
split_data = data.train_test_split(test_size=0.15, stratify_by_column="label", seed=42)
cv_train = split_data["train"]      # The 85% we will use for the 5-Fold Loop
test = split_data["test"]

print(f"Total images: {len(data)}")
print(f"Images for Cross-Validation: {len(cv_train)}")
print(f"Images locked in the Test Vault: {len(test)}")

In [ ]:
labels = cv_train.features["label"].names
id2label = {str(i): c for i, c in enumerate(labels)}
label2id = {c: str(i) for i, c in enumerate(labels)}
all_numerical_labels = cv_train["label"]

print(f"Classes found: {labels}")

## Setup and training for the default image resolution (224x224)

In the project we used the pre-trained EfficientNet-B0 architecture from Hugging Face as the base model for image classification. EfficientNet-B0 was selected because it offers a good balance between accuracy and computational requirements. Instead of building and training a model from the beginning, a transfer learning approach was used. This means that a model already trained on a large image dataset was reused and adapted for the garbage classification task. 

Due to the computational complexity of the training process, particularly when performing 5-fold cross-validation and experimenting with different image resolutions, local execution became impractical. Training required substantial processing power and longer execution times, therefore cloud-based resources were used to support the experiments. To enable execution in the cloud environment, the notebook workflow was converted into a standalone Trash_classification.py script. This allowed the training process to run more efficiently outside the notebook environment and simplified execution on remote computational resources. Converting the workflow into a script also improved reproducibility and made it easier to manage long-running training jobs.

During training, outputs from each cross-validation fold were automatically saved into separate folders. This included model checkpoints, training logs, evaluation metrics, and configuration files generated throughout the learning process.

In [ ]:
model_name = "google/efficientnet-b0" 
image_processor = AutoImageProcessor.from_pretrained(model_name)

In [ ]:
augmentations = Compose([
    RandomHorizontalFlip(),
    RandomRotation(20),
    ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1)
])

# Transform for TRAINING with augmentations
def train_transforms(batch):
    augmented_images = [augmentations(x.convert("RGB")) for x in batch["image"]] #  Apply the random flips and colors to the raw PIL images
    inputs = image_processor(augmented_images, return_tensors="pt") # Resize, Tensor, and Normalize
    inputs["label"] = batch["label"]
    return inputs

# Transform for VALIDATION without augmentations
def val_transforms(batch):
    inputs = image_processor([x.convert("RGB") for x in batch["image"]], return_tensors="pt")
    inputs["label"] = batch["label"]
    return inputs

def collate_fn(batch):
    return {
        'pixel_values': torch.stack([x['pixel_values'] for x in batch]),
        'labels': torch.tensor([x['label'] for x in batch])
    }

# Metric calculation
accuracy = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    predictions, true_labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=true_labels)


In [ ]:
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

fold_accuracies = [] 
best_overall_accuracy = 0.0  
best_fold = 0
oof_preds = []
oof_labels = []

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(all_numerical_labels)), all_numerical_labels)):
    print(f"\n================================")
    print(f"       STARTING FOLD {fold + 1}/{n_splits}")
    print(f"================================")

    # Create  train/val subsets for this round
    fold_train_data = cv_train.select(train_idx).with_transform(train_transforms)
    fold_val_data = cv_train.select(val_idx).with_transform(val_transforms)

    
    model = AutoModelForImageClassification.from_pretrained(
        model_name,
        num_labels=len(labels), # num_labels is set to the number of classes in our dataset
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True  # allows loading pretrained weights even if the classifier head size doesn’t match your number of classes
    )

    # Setup Trainer for this fold
    training_args = TrainingArguments(
        output_dir=f"./garbage_model_fold_{fold + 1}", # Save each fold in a separate folder
        remove_unused_columns=False, # preventing deleting images
        eval_strategy="epoch", # Evaluate at the end of every epoch
        save_strategy="epoch", # Save model at the end of every epoch
        learning_rate=5e-5,
        per_device_train_batch_size=16, # batch size for training
        gradient_accumulation_steps=2, # accumulate gradients over 2 steps to effectively have a batch size of 64 without OOM errors
        per_device_eval_batch_size=16, # batch size for test
        num_train_epochs=15, # 3  number of training epochs
        warmup_ratio=0.1, # 10% of training steps will be used for a linear warmup of the learning rate
        logging_steps=10, # controls how often the training process prints out progress or metrics
        load_best_model_at_end=True,
        metric_for_best_model="accuracy"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        data_collator=collate_fn,
        train_dataset=fold_train_data,
        eval_dataset=fold_val_data,
        processing_class=image_processor,
        compute_metrics=compute_metrics,
    )
    trainer.train() # Train the Fold
    
    # Get final validation score per fold
    prediction_output = trainer.predict(fold_val_data)
    fold_acc = prediction_output.metrics['test_accuracy'] 
    fold_accuracies.append(fold_acc)
    print(f"\n>>> Fold {fold + 1} Accuracy: {fold_acc * 100:.2f}%\n")

    # Grab the winning guesses (argmax) and the true answers
    batch_preds = np.argmax(prediction_output.predictions, axis=1)
    batch_labels = prediction_output.label_ids
    
    # Store them in our global OOF lists
    oof_preds.extend(batch_preds)
    oof_labels.extend(batch_labels)

    if fold_acc > best_overall_accuracy:
        print(f"Best model so far found in Fold {fold + 1} (Accuracy: {fold_acc * 100:.2f}%). Saving...")
        best_overall_accuracy = fold_acc
        best_fold = fold + 1
        trainer.save_model("./best_garbage_classifier")
        image_processor.save_pretrained("./best_garbage_classifier")

    # Free up computer memory before starting the next fold
    del model, trainer, training_args
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print("\n================================")
print("      CROSS-VALIDATION COMPLETE")
print("================================")
for i, acc in enumerate(fold_accuracies):
    print(f"Fold {i + 1}: {acc * 100:.2f}%")
print(f"\nAverage Accuracy across all {n_splits} folds: {np.mean(fold_accuracies) * 100:.2f}%")
print(f"Standard Deviation: ±{np.std(fold_accuracies) * 100:.2f}%")

print(f"\nThe best model was Fold {best_fold} with {best_overall_accuracy * 100:.2f}% accuracy.")
print("It has been saved to the folder: './best_garbage_classifier'")

In [ ]:
plt.figure(figsize=(16, 6))
ax1 = plt.subplot(1, 2, 1)
ax2 = plt.subplot(1, 2, 2)

folds = 5
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

for i in range(1, folds + 1):
    fold_dir = f"./garbage_model_fold_{i}"
    
    if not os.path.exists(fold_dir):
        print(f"Could not find folder: {fold_dir}")
        continue

    # logs inside 'checkpoint-XXX' folders
    checkpoints = [d for d in os.listdir(fold_dir) if d.startswith("checkpoint")]
    if not checkpoints:
        print(f"No checkpoints found in {fold_dir}")
        continue

    checkpoints.sort(key=lambda x: int(x.split("-")[1]))
    latest_checkpoint = checkpoints[-1]
    
    state_path = os.path.join(fold_dir, latest_checkpoint, "trainer_state.json")
    
    if os.path.exists(state_path):
        with open(state_path, "r") as f:
            state = json.load(f)
            
        log_history = state["log_history"]
        
        # Loss for every 10 steps (logging_steps=10)
        train_steps = [log["step"] for log in log_history if "loss" in log]
        train_loss = [log["loss"] for log in log_history if "loss" in log]
        
        # Accuracy per each epoch
        eval_epochs = [log["epoch"] for log in log_history if "eval_accuracy" in log]
        eval_acc = [log["eval_accuracy"] * 100 for log in log_history if "eval_accuracy" in log] 
    
        ax1.plot(train_steps, train_loss, color=colors[i-1], alpha=0.8, label=f"Fold {i}")
        ax2.plot(eval_epochs, eval_acc, color=colors[i-1], marker='o', linewidth=2, label=f"Fold {i}")

# Training Loss
ax1.set_title("1. Training Loss", fontsize=14, fontweight='bold')
ax1.set_xlabel("Training Steps", fontsize=12)
ax1.set_ylabel("Loss (Error Rate)", fontsize=12)
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend()

# Validation Accuracy 
ax2.set_title("2. Validation Accuracy", fontsize=14, fontweight='bold')
ax2.set_xlabel("Epochs", fontsize=12)
ax2.set_ylabel("Accuracy (%)", fontsize=12)
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend()

plt.tight_layout()
plt.show()

Accuracy was selected as the main evaluation metric. For each prediction, the model outputs probabilities for all classes, and the class with the highest value is selected as the final prediction. This prediction is then compared with the true label. The model was trained using 5-fold stratified cross-validation.Stratification helped keep the class distribution similar in every fold. For each fold, a new EfficientNet-B0 model was loaded with a classification layer adjusted to the 11 garbage categories. The model was trained for 15 epoch.

Two plots were created: one showing training loss over training steps and another showing validation accuracy over epochs.

In [ ]:
from transformers.utils.notebook import NotebookProgressCallback

In [ ]:
best_model = AutoModelForImageClassification.from_pretrained("./best_garbage_classifier")
best_processor = AutoImageProcessor.from_pretrained("./best_garbage_classifier")

test_prep = test.with_transform(val_transforms)

test_args = TrainingArguments(
    output_dir="./final_test_results",
    remove_unused_columns=False, 
    per_device_eval_batch_size=16,
    report_to="none" 
)

# Short evaluation
final_trainer = Trainer(
    model=best_model,
    args=test_args,
    data_collator=collate_fn,
    eval_dataset=test_prep,
    processing_class=best_processor,
    compute_metrics=compute_metrics,
)

final_trainer.remove_callback(NotebookProgressCallback)
print(f"Testing against {len(test_prep)} unseen vault images")
final_results = final_trainer.evaluate()
final_accuracy = final_results['eval_accuracy'] * 100
print(f"\n========================================")
print(f"Final test vault accuracy: {final_accuracy:.2f}%")


Finally, the model generated predictions for all unseen images and calculated the final accuracy score. This provided an objective estimate of model performance and generalization ability on data that was never used during training.

## Setup and training for the higher image resolution (448x448)

The same evaluation procedure was repeated for the 448×448 model to enable a fair comparison between image resolutions. Identical preprocessing steps, evaluation settings, and performance metrics were applied to ensure consistency across experiments.

In [ ]:
image_processor = AutoImageProcessor.from_pretrained(model_name)
image_processor.size = {"height": 448, "width": 448}

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

fold_accuracies = [] 
best_overall_accuracy = 0.0  
best_fold = 0
oof_preds = []
oof_labels = []

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(all_numerical_labels)), all_numerical_labels)):
    print(f"\n================================")
    print(f"       STARTING FOLD {fold + 1}/{n_splits}")
    print(f"================================")

    # Create  train/val subsets for this round
    fold_train_data = cv_train.select(train_idx).with_transform(train_transforms)
    fold_val_data = cv_train.select(val_idx).with_transform(val_transforms)

    
    model = AutoModelForImageClassification.from_pretrained(
        model_name,
        num_labels=len(labels), # num_labels is set to the number of classes in our dataset
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True  # allows loading pretrained weights even if the classifier head size doesn’t match your number of classes
    )

    # Setup Trainer for this fold
    training_args = TrainingArguments(
        output_dir=f"./garbage_model_448_fold_{fold + 1}", 
        remove_unused_columns=False, 
        eval_strategy="epoch", 
        save_strategy="epoch", 
        learning_rate=5e-5,
        per_device_train_batch_size=8, # reduced from 16
        gradient_accumulation_steps=4,  # reduced from 4
        per_device_eval_batch_size=8,   # reduced from 16
        num_train_epochs=15,      
        warmup_ratio=0.1, 
        logging_steps=10, 
        load_best_model_at_end=True,
        metric_for_best_model="accuracy"
    )


    trainer = Trainer(
        model=model,
        args=training_args,
        data_collator=collate_fn,
        train_dataset=fold_train_data,
        eval_dataset=fold_val_data,
        processing_class=image_processor,
        compute_metrics=compute_metrics,
    )
    trainer.train() # Train the Fold
    
    # Get final validation score per fold
    prediction_output = trainer.predict(fold_val_data)
    fold_acc = prediction_output.metrics['test_accuracy'] 
    fold_accuracies.append(fold_acc)
    print(f"\n>>> Fold {fold + 1} Accuracy: {fold_acc * 100:.2f}%\n")

    # Grab the winning guesses (argmax) and the true answers
    batch_preds = np.argmax(prediction_output.predictions, axis=1)
    batch_labels = prediction_output.label_ids
    
    # Store them in our global OOF lists
    oof_preds.extend(batch_preds)
    oof_labels.extend(batch_labels)

    if fold_acc > best_overall_accuracy:
        print(f"Best 448px model so far found in Fold {fold + 1} (Accuracy: {fold_acc * 100:.2f}%). Saving...")
        best_overall_accuracy = fold_acc
        best_fold = fold + 1
        trainer.save_model("./model_448px_experiment")
        image_processor.save_pretrained("./model_448px_experiment")

    # Free up computer memory before starting the next fold
    del model, trainer, training_args
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    

print("\n================================")
print("      CROSS-VALIDATION COMPLETE")
print("================================")
for i, acc in enumerate(fold_accuracies):
    print(f"Fold {i + 1}: {acc * 100:.2f}%")
print(f"\nAverage Accuracy across all {n_splits} folds: {np.mean(fold_accuracies) * 100:.2f}%")
print(f"Standard Deviation: ±{np.std(fold_accuracies) * 100:.2f}%")

print(f"\nThe best model was Fold {best_fold} with {best_overall_accuracy * 100:.2f}% accuracy.")
print("It has been saved to the folder: './model_448px_experiment'")

The training phase focused on building a garbage image classification model using a transfer learning approach based on EfficientNet-B0. After preparing the dataset, a 15% hold-out test set was separated and reserved for final evaluation, while the remaining 85% was used during training and validation. To improve model robustness and reduce overfitting, data augmentation techniques such as random flipping, rotation, and color adjustments were applied to training images. The model was trained using 5-fold stratified cross-validation, ensuring that each class maintained a similar distribution across folds and allowing performance to be evaluated more reliably.

Two experiments were performed using image resolutions of 224×224 and 448×448 to investigate the impact of image size on model performance. During training, outputs from each fold, including checkpoints and logs, were saved into separate folders for later analysis. The results showed that although the 448×448 model achieved very low training loss, it displayed less stable validation performance and signs of overfitting. In comparison, the 224×224 model produced more consistent results across folds and demonstrated better generalization, making it the selected model for final evaluation.

In the Part2_Test_models.ipynb notebook we perform the testing part.